<a href="https://colab.research.google.com/github/AzureSolo/Land-suitability-project/blob/main/scripts/01_Clean_Data_for_Shareability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Set up connections to Google Drive and libraries

In [ ]:
# Run this cell with a Python runtime only.
# Switch back to R before running any cell below.
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#switch runtime to R. This is a MUST to run the next steps
setwd("/content/drive/MyDrive/land_suitability_project")

In [ ]:
#Connect to libraries
library(readr) # read_csv(), write_csv()
library(dplyr) # rename(), mutate(), filter(), left_join(), anti_join()
library(tidyr) # pivot_longer()


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




Data Cleaning Prep

In [ ]:
#Set up a reference variable for shorthand
files <- list.files(
  path = "raw_data/borough_files/split",
  pattern = "\\.csv$",
  full.names = TRUE
)

In [ ]:
#Establish the data cleaning step variable to allow for a future loop through the ~20 raw data sheets. This is faster than repeating each workflow individually.

data_cleaning_step  <- function(file_path){

  # 1. Read file
  pluto_raw <- read_csv(
    file_path,
    show_col_types = FALSE)

  # 2. Rename column heading to improve data usability.
  #Data entires are not touched yet
  pluto_clean <- pluto_raw %>% #pluto is the datasource name for context on my naming
    rename(
      # 2.1 Parcel Info
      borough_name = borough,
      tax_block = `Tax block`,
      tax_lot = `Tax lot`,
      parcel_id = BBL,
      condominium_number = condono,

      # 2.2 Location
      community_board = `community board`,
      census_tract_2010 = `census tract 2010`,
      census_block_2010 = cb2010,
      school_district = schooldist,
      city_council_district = `council district`,
      zip_code = postcode,
      fire_company = firecomp,
      police_precinct = policeprct,
      health_area = healtharea,
      sanitation_borough = sanitboro,
      sanitation_subdistrict = sanitsub,
      street_address = address,
      x_coordinate = xcoord,
      y_coordinate = ycoord,

      # 2.3 Zoning
      zoning_district_1 = zonedist1,
      zoning_district_2 = zonedist2,
      zoning_district_3 = zonedist3,
      zoning_district_4 = zonedist4,
      zoning_overlay_1 = overlay1,
      zoning_overlay_2 = overlay2,
      special_district_1 = spdist1,
      special_district_2 = spdist2,
      special_district_3 = spdist3,
      limited_height_district = ltdheight,
      split_zoning_flag = splitzone,

      # 2.4 Land Use
      building_class = bldgclass,
      land_use = landuse,
      easement_count = easements,

      # 2.5 Ownership
      owner_type = ownertype,
      owner_name = ownername,

      # 2.6 Area
      #Include the unit of measure for easy reference
      lot_area_sqft = lotarea,
      building_area_sqft = bldgarea,
      commercial_area_sqft = comarea,
      residential_area_sqft = resarea,
      office_area_sqft = officearea,
      retail_area_sqft = retailarea,
      garage_area_sqft = garagearea,
      storage_area_sqft = strgearea,
      factory_area_sqft = factryarea,
      other_area_sqft = otherarea,

      # 2.7 Building Characteristics
      number_of_buildings = numbldgs,
      number_of_floors = numfloors,
      residential_units = unitsres,
      total_units = unitstotal,
      lot_frontage_ft = lotfront,
      lot_depth_ft = lotdepth,
      building_frontage_ft = bldgfront,
      building_depth_ft = bldgdepth,

      # 2.8 Assessment
      assessed_land_value = assessland,
      assessed_total_value = assesstot,
      exempt_value = exempttot,

      # 2.9 Building Age
      year_built = yearbuilt,
      major_alteration_year_1 = yearalter1,
      major_alteration_year_2 = yearalter2,

      # 2.10 Historic
      historic_district = histdist,
      landmark_status = landmark,

      # 2.11 FAR
      #FAR stands for Floor Area Ratio. This acroymn is very common in urban planning so it left undefined.
      built_far = builtfar,
      residential_far = residfar,
      commercial_far = commfar,
      facility_far = facilfar,
      affordable_housing_far = affresfar,
      manufacturing_far = mnffar
    )
#Renaming note: Some columns were left alone. I don't think they were will be used at this point; however, I am not analyzing the data yet so I didn' remove those columns.


  # 3. Remove columns that contain only NA values
  #Table is very long and wide. Removing empty/NA columns may help alleviate
  pluto_clean <- pluto_clean %>%
    select(where(~ !all(is.na(.))))

  # 4. Convert blanks to NA
  #I probably could move this to be prior to Step 3/Removing.
  pluto_clean <- pluto_clean %>%
    mutate(
      across(
        where(is.character),
        ~ na_if(.x, "")
      ))

  # 5. Remove duplicate parcel IDs
  pluto_clean <- pluto_clean %>%
    distinct(parcel_id, .keep_all = TRUE)

  # 6. Convert some columns to character
  pluto_clean <- pluto_clean %>%
    mutate(
      across(
        c(
          tax_lot,
          community_board,
          census_tract_2010,
          census_block_2010,
          school_district,
          city_council_district,
          zip_code,
          police_precinct,
          health_area,
          sanitation_borough,
          land_use,
          year_built,
          major_alteration_year_1,
          major_alteration_year_2,
          parcel_id
        ),
        as.character
      )
    )

  # 7. improve land use coding in fields
  #Numbers don't mean anything. Using the actual land use category per NYC is better for data comprehension.
  pluto_clean <- pluto_clean %>%
    mutate(
      land_use = case_when(
        land_use == "1"  ~ "One and Two Family Buildings",
        land_use == "2"  ~ "Multi-Family Walk-Up Residences",
        land_use == "3"  ~ "Multi-Family Elevator Residences",
        land_use == "4"  ~ "Mixed Residential and Commercial",
        land_use == "5"  ~ "Commercial Uses",
        land_use == "6"  ~ "Industrial and Manufacturing",
        land_use == "7"  ~ "Transportation and Utility",
        land_use == "8"  ~ "Public Facilities and Institutions",
        land_use == "9"  ~ "Open Space and Recreation",
        land_use == "10" ~ "Parking",
        land_use == "11" ~ "Vacant Land",
        TRUE ~ land_use
      ))

  # 8. Move parcel_id to front
  #parcel_id will likely be the main join field during the analysis step
  pluto_clean <- pluto_clean %>%
    relocate(parcel_id, .before = 1)
  return(pluto_clean)
}

Write each cleaned file as a csv.

In [ ]:
#Loop through the raw borough files to produce cleaned data
for(file in files){
  cleaned <- data_cleaning_step(file) #files is the variable name that "directs" to the location of the raw data files

  output_file <- file.path(
    "clean_data/borough_files",
    gsub(
      "raw_",
      "01_clean_", #replacesthe raw_ prefix with 01_clean_. 01_clean refers to the script being used for the clean data.
      basename(file)))
#Produce/save datasets as CSV files
  write_csv(cleaned, output_file)

}

ERROR: Error: object 'files' not found
